# Data Engineering


In [1]:
# import relevant libraries

import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

## Purpose

The purpose of this notebook is to improve and optimize the raw datset so as to allow efficient loading and usage of data. The first step is to load the datawe are planning on using and previewing it and the data types they hold, then preview the ranges the data actually uses to see if there is a more efficient storage type. 

In [2]:
# import relevant data tables

campaigns = pd.read_csv(r'C:\Users\Tangy\Downloads\archive (2)\campaigns.csv')
customers = pd.read_csv(r'C:\Users\Tangy\Downloads\archive (2)\customers.csv')
products = pd.read_csv(r'C:\Users\Tangy\Downloads\archive (2)\products.csv')
transactions = pd.read_csv(r'C:\Users\Tangy\Downloads\archive (2)\transactions.csv')
events = pd.read_csv(r'C:\Users\Tangy\Downloads\archive (2)\events.csv')

In [3]:
# preview datasets and data types
for df in [campaigns, customers, products, transactions]:
    print(df.head())
    print(df.dtypes)
    

   campaign_id      channel     objective  start_date    end_date  \
0            1  Paid Search    Cross-sell  2021-10-25  2021-11-26   
1            2        Email     Retention  2021-10-24  2021-12-24   
2            3        Email  Reactivation  2023-10-08  2023-11-30   
3            4      Display  Reactivation  2022-07-25  2022-10-07   
4            5       Social   Acquisition  2022-07-09  2022-09-29   

  target_segment  expected_uplift  
0   Deal Seekers            0.022  
1   Deal Seekers            0.116  
2     Churn Risk            0.100  
3   Deal Seekers            0.111  
4  New Customers            0.144  
campaign_id          int64
channel             object
objective           object
start_date          object
end_date            object
target_segment      object
expected_uplift    float64
dtype: object
   customer_id signup_date country  age  gender loyalty_tier  \
0            1  2021-04-08      BR   48    Male       Bronze   
1            2  2023-04-28      IN   3

In [4]:
# check max values of numerical columns to verify if integer data type is necessary for the existing range
events[['event_id', 'customer_id', 'product_id', 'session_id', 'session_duration_sec']].max()

event_id                2000000.0
customer_id              100000.0
product_id                 2000.0
session_id               666666.0
session_duration_sec       7533.8
dtype: float64

In [5]:
# cast types to be memory efficient
ca_dtypes = {
    'campaign_id': 'int32',
    'channel': 'category',
    'objective': 'category',
    'start_date': 'object',
    'end_date': 'object',
    'target_segment': 'category',
    'expected_uplift': 'float32'
}

cu_dtypes = {
    'customer_id':             'int32',
    'signup_date':            'object',
    'country':                'category',
    'age':                     'int32',
    'gender':                 'category',
    'loyalty_tier':           'category',
    'acquisition_channel':    'category'
}

pr_dtypes = {
    'product_id':       'int32',
    'category':        'category',
    'brand':           'category',
    'base_price':     'float32',
    'launch_date':     'object',
    'is_premium':       'bool'
}

tr_dtypes = {
    'transaction_id':        'int32',
    'timestamp':            'object',
    'customer_id':           'int32',
    'product_id':          'float32',
    'quantity':              'int32',
    'discount_applied':    'float32',
    'gross_revenue':       'float32',
    'campaign_id':           'int32',
    'refund_flag':           'bool'
}

ev_dtypes = {
    'event_id': 'int32',
    'timestamp': 'str',
    'customer_id': 'int32',
    'session_id': 'int32',
    'event_type': 'category',
    'product_id': 'float32',
    'device_type': 'category',
    'traffic_source': 'category',
    'campaign_id': 'int32',
    'page_category': 'category',
    'session_duration_sec': 'float32',
    'experiment_group': 'category'
}

In [6]:
# re-import data sets with the efficiency improved memory types
campaigns_idf = pd.read_csv(r'C:\Users\Tangy\Downloads\archive (2)\campaigns.csv', dtype=ca_dtypes)
customers_idf = pd.read_csv(r'C:\Users\Tangy\Downloads\archive (2)\customers.csv', dtype=cu_dtypes)
products_idf = pd.read_csv(r'C:\Users\Tangy\Downloads\archive (2)\products.csv', dtype=pr_dtypes)
transactions_idf = pd.read_csv(r'C:\Users\Tangy\Downloads\archive (2)\transactions.csv', dtype=tr_dtypes)
events_idf = pd.read_csv(r'C:\Users\Tangy\Downloads\archive (2)\events.csv', dtype=ev_dtypes)

In [7]:
for df in [campaigns_idf, customers_idf, products_idf, events_idf]:
    print(df.describe(include='category'))
    print(df.isnull().sum())

          channel     objective target_segment
count          50            50             50
unique          5             4              5
top     Affiliate  Reactivation  New Customers
freq           11            15             12
campaign_id        0
channel            0
objective          0
start_date         0
end_date           0
target_segment     0
expected_uplift    0
dtype: int64
       country  gender loyalty_tier acquisition_channel
count   100000  100000       100000              100000
unique       7       3            4                   5
top         US    Male       Bronze             Organic
freq     34931   48054        60276               30200
customer_id            0
signup_date            0
country                0
age                    0
gender                 0
loyalty_tier           0
acquisition_channel    0
dtype: int64
           category    brand
count          2000     2000
unique            6      100
top     Electronics  Brand_7
freq            455  

We also need to clean and check integrity of data before use, so we preview the null values to see if there are rows to drop before analysis.

In [8]:
# preview null values in set
print(transactions_idf.isnull().sum())

transaction_id          0
timestamp               0
customer_id             0
product_id          10449
quantity                0
discount_applied        0
gross_revenue       10449
campaign_id             0
refund_flag             0
dtype: int64


There are an equal amount of null values in the product_id and gross_revenue categories. The planned analysis includes these columns, but these values will only be dropped as necessary since correlations between other columns are more important in some cases. However, I changed the values from null to zero so those rows can be in line with all parts of the analysis. An additional category for these cases, being a 0 product_id and an "unknown" type for non-numerical categories, are added so as to make these cases distinct. 

In [9]:
if 'Unknown' not in events_idf['device_type'].cat.categories:
    events_idf.loc[:, 'device_type'] = events_idf['device_type'].cat.add_categories('Unknown')
events_idf.loc[events_idf['device_type'].isna(), 'device_type'] = 'Unknown'

events_idf.loc[events_idf['product_id'].isna(), 'product_id'] = 0

# adjust nulls in transactions
transactions_idf.loc[transactions_idf['product_id'].isna(), 'product_id'] = 0
transactions_idf.loc[transactions_idf['gross_revenue'].isna(), 'gross_revenue'] = 0

C:\Users\Tangy\AppData\Local\Temp\ipykernel_27040\2820577990.py:2: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['desktop', 'desktop', 'mobile', 'desktop', 'desktop', ..., 'mobile', 'desktop', 'mobile', 'mobile', 'mobile']
Length: 2000000
Categories (4, object): ['desktop', 'mobile', 'tablet', 'Unknown']' has dtype incompatible with category, please explicitly cast to a compatible dtype first.
  events_idf.loc[:, 'device_type'] = events_idf['device_type'].cat.add_categories('Unknown')


I also verify that the changes in memory types and cleaning actually made a difference in memory by checking the memory usage of the data sets (the original and the new version) before exporting to use for analysis. 

In [10]:
# check initial dataset memory usage
itot = 0
for df in [campaigns, customers, products, transactions, events]:
    itot = itot + df.memory_usage(deep=True).sum()
    print(itot)

15595
29183110
29575263
43188159
825810567


In [11]:
# check initial dataset memory usage
otot = 0
for df in [campaigns_idf, customers_idf, products_idf, transactions_idf, events_idf]:
    otot = otot + df.memory_usage(deep=True).sum()
    print(otot)

7912
7109817
7260316
17263767
211266323


In [12]:
# export optimized datasets
campaigns_idf.to_csv(r'C:\Users\Tangy\Downloads\archive (2)\campaigns_idf.csv', index=False)
customers_idf.to_csv(r'C:\Users\Tangy\Downloads\archive (2)\customers_idf.csv', index=False)
products_idf.to_csv(r'C:\Users\Tangy\Downloads\archive (2)\products_idf.csv', index=False)
transactions_idf.to_csv(r'C:\Users\Tangy\Downloads\archive (2)\transactions_idf.csv', index=False)
events_idf.to_csv(r'C:\Users\Tangy\Downloads\archive (2)\events_idf.csv', index=False)